# PBIT Safety — Final Test Set Evaluation

**Objective**: evaluate and compare the F1-tuned and CMR-tuned models on the held-out test set.

Both models are trained once on the **full training set** (1,196 real + augmented F04/F05/F06 to 160)
and evaluated once on the **292 held-out experiments** from 11 unseen serial numbers.

| Model | Params | f05_boost | sev3_boost | Config |
|-------|--------|----------:|----------:|--------|
| **F1-tuned** | `best_lgbm_params.json` | 6.98× | 1.0× | Aug + Freq×Sev |
| **CMR-tuned** | `best_lgbm_cmr_params.json` | 2.94× | 19.48× | Aug + Freq×Sev |

Results saved to `data/final_safety_eval_results.csv`.

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks_safety':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.metrics import confusion_matrix

from pbit.modeling import (
    META_COLS, FAULT_ORDER, SEVERITY_MAP,
    compute_sample_weights, evaluate_predictions, compute_safety_metrics,
)
from pbit.data_validation import load_long_dataset
from pbit.augmentation import augment_training_set
from pbit.features import extract_features

TRAIN_PATH      = PROJECT_ROOT / 'data' / 'train_real.csv'
TEST_PATH       = PROJECT_ROOT / 'data' / 'test_real.csv'
LONG_PATH       = PROJECT_ROOT / 'data' / 'PBIT_atuador_DESCARAC_r1_long_clean.csv'
F1_PARAMS_PATH  = PROJECT_ROOT / 'data' / 'best_lgbm_params.json'
CMR_PARAMS_PATH = PROJECT_ROOT / 'data' / 'best_lgbm_cmr_params.json'
SEED_CSV_PATH   = PROJECT_ROOT / 'data' / 'cv_seed_stability.csv'
RESULTS_PATH    = PROJECT_ROOT / 'data' / 'final_safety_eval_results.csv'

PROJECT_ROOT

## Load Data & Params

In [ ]:
train_df  = pd.read_csv(TRAIN_PATH)
test_df   = pd.read_csv(TEST_PATH)
long_df   = load_long_dataset(LONG_PATH)
feat_cols = [c for c in train_df.columns if c not in META_COLS]

# F1-tuned params
with open(F1_PARAMS_PATH) as f:
    f1_all = json.load(f)
f1_f05_boost = f1_all.pop('f05_boost')
f1_params    = f1_all

# CMR-tuned params
with open(CMR_PARAMS_PATH) as f:
    cmr_all = json.load(f)
cmr_sev3_boost = cmr_all.pop('sev3_boost')
cmr_f05_boost  = cmr_all.pop('f05_boost')
cmr_params     = cmr_all

print(f'Train : {len(train_df):,} experiments x {len(feat_cols)} features')
print(f'Test  : {len(test_df):,} experiments')
print()
print(f'F1-tuned   : f05_boost={f1_f05_boost:.3f}')
print(f'CMR-tuned  : f05_boost={cmr_f05_boost:.3f}  sev3_boost={cmr_sev3_boost:.3f}')
print()
print('Test class distribution:')
print(test_df['Fault_Mode'].value_counts().reindex(FAULT_ORDER).to_string())

## Build Full Training Set (Real + Augmented)

In [ ]:
train_exp_ids = set(train_df['Experiment_Index'].tolist())
aug_long = augment_training_set(
    long_df, train_exp_ids,
    target_per_class=160, noise_scale=0.01, random_state=42,
)
aug_feat = extract_features(aug_long)
train_full = pd.concat([train_df, aug_feat], ignore_index=True)

X_train = train_full[feat_cols].values.astype(float)
y_train = train_full['Fault_Mode'].values
X_test  = test_df[feat_cols].values.astype(float)
y_test  = test_df['Fault_Mode'].values

print(f'Full training set: {len(train_full):,} samples')
print('Augmented class counts:')
print(pd.Series(y_train).value_counts().reindex(FAULT_ORDER).to_string())

## Train Both Models

In [ ]:
# --- F1-tuned model ---
sw_f1 = compute_sample_weights(y_train, 'freq_sev')
sw_f1[y_train == 'F05'] *= f1_f05_boost

t0 = time.time()
model_f1 = lgb.LGBMClassifier(**f1_params)
model_f1.fit(X_train, y_train, sample_weight=sw_f1)
print(f'F1-tuned  trained in {time.time()-t0:.1f}s')

# --- CMR-tuned model ---
sw_cmr = compute_sample_weights(y_train, 'freq_sev')
sw_cmr[y_train == 'F05'] *= cmr_f05_boost
sw_cmr[np.isin(y_train, ['F03', 'F06'])] *= cmr_sev3_boost

t0 = time.time()
model_cmr = lgb.LGBMClassifier(**cmr_params)
model_cmr.fit(X_train, y_train, sample_weight=sw_cmr)
print(f'CMR-tuned trained in {time.time()-t0:.1f}s')

# Predictions
y_pred_f1  = model_f1.predict(X_test)
y_pred_cmr = model_cmr.predict(X_test)

print('\nPredictions done.')

## Test Set Metrics Comparison

In [ ]:
m_f1  = evaluate_predictions(y_test, y_pred_f1)
sm_f1 = compute_safety_metrics(y_test, y_pred_f1)

m_cmr  = evaluate_predictions(y_test, y_pred_cmr)
sm_cmr = compute_safety_metrics(y_test, y_pred_cmr)

print('=== Test set results ===')
print(f"{'Metric':<25} {'F1-tuned':>12} {'CMR-tuned':>12} {'Delta':>10}")
print('-' * 62)
rows = [
    ('Macro F1',       'macro_f1'),
    ('Balanced Acc',   'balanced_accuracy'),
    ('F01 Recall',     'f01_recall'),
    ('F02 Recall',     'f02_recall'),
    ('F03 Recall',     'f03_recall'),
    ('F04 Recall',     'f04_recall'),
    ('F05 Recall',     'f05_recall'),
    ('F06 Recall',     'f06_recall'),
    ('Sev3 Recall',    'sev3_recall'),
]
for label, key in rows:
    v1 = m_f1.get(key, float('nan'))
    v2 = m_cmr.get(key, float('nan'))
    print(f"{label:<25} {v1:>12.4f} {v2:>12.4f} {v2-v1:>+10.4f}")

print()
print(f"{'Safety metric':<25} {'F1-tuned':>12} {'CMR-tuned':>12} {'Delta':>10}")
print('-' * 62)
for label, sm1_key, sm2_key in [
    ('CMR',            'cmr',       'cmr'),
    ('CMR count',      'cmr_count', 'cmr_count'),
    ('Crit total',     'crit_total','crit_total'),
    ('Downward Risk',  'down_risk', 'down_risk'),
    ('Severity Error', 'sev_err',   'sev_err'),
]:
    v1 = sm_f1.get(sm1_key, float('nan'))
    v2 = sm_cmr.get(sm2_key, float('nan'))
    if isinstance(v1, int) or (isinstance(v1, float) and v1 == int(v1) and label in ('CMR count','Crit total')):
        print(f"{label:<25} {int(v1):>12} {int(v2):>12} {int(v2)-int(v1):>+10}")
    else:
        print(f"{label:<25} {v1:>12.4f} {v2:>12.4f} {v2-v1:>+10.4f}")

## CMR Breakdown — Which Sev3 Samples Were Predicted as Sev0

In [ ]:
SEV0 = {'N01', 'N02'}
SEV3 = {'F03', 'F06'}

for name, y_pred in [('F1-tuned', y_pred_f1), ('CMR-tuned', y_pred_cmr)]:
    crit_mask = np.array([c in SEV3 for c in y_test])
    sev0_mask = np.array([c in SEV0 for c in y_pred])
    miss_idx  = np.where(crit_mask & sev0_mask)[0]

    print(f'{name}: {len(miss_idx)} critical misses ({100*len(miss_idx)/crit_mask.sum():.1f}% of {crit_mask.sum()} Sev3 samples)')
    if len(miss_idx) > 0:
        pairs = pd.Series(
            [f'{y_test[i]} → {y_pred[i]}' for i in miss_idx]
        ).value_counts()
        for pair, cnt in pairs.items():
            print(f'  {pair}: {cnt}')
    else:
        print('  No critical misses — CMR = 0')
    print()

## Confusion Matrices

In [ ]:
present_labels = [c for c in FAULT_ORDER if c in set(y_test)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, name, y_pred in [
    (axes[0], 'F1-tuned',  y_pred_f1),
    (axes[1], 'CMR-tuned', y_pred_cmr),
]:
    cm = confusion_matrix(y_test, y_pred, labels=present_labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    annot = np.array([[f'{cm[i,j]}\n({cm_norm[i,j]:.0%})'
                       for j in range(len(present_labels))]
                      for i in range(len(present_labels))])
    sns.heatmap(
        cm_norm, annot=annot, fmt='', cmap='Blues',
        xticklabels=present_labels, yticklabels=present_labels,
        vmin=0, vmax=1, linewidths=0.3, ax=ax,
    )
    # Highlight Sev3→Sev0 cells in red
    for i, true_c in enumerate(present_labels):
        for j, pred_c in enumerate(present_labels):
            if true_c in SEV3 and pred_c in SEV0:
                ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                           edgecolor='red', lw=2.5))
    m  = evaluate_predictions(y_test, y_pred)
    sm = compute_safety_metrics(y_test, y_pred)
    ax.set_title(f'{name}\nMacro F1={m["macro_f1"]:.3f}  CMR={sm["cmr"]:.4f}  DR={sm["down_risk"]:.4f}',
                 fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

fig.suptitle('Confusion matrices — test set (292 experiments, 11 unseen SNs)\n'
             'Red border = Sev3→Sev0 critical miss', fontsize=11)
plt.tight_layout()
plt.show()

## Severity-Level Confusion

In [ ]:
sev_levels = sorted(set(SEVERITY_MAP.values()))
sev_true = np.array([SEVERITY_MAP.get(c, 0) for c in y_test])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name, y_pred in [
    (axes[0], 'F1-tuned',  y_pred_f1),
    (axes[1], 'CMR-tuned', y_pred_cmr),
]:
    sev_pred = np.array([SEVERITY_MAP.get(c, 0) for c in y_pred])
    cm_sev = confusion_matrix(sev_true, sev_pred, labels=sev_levels)
    cm_sev_norm = cm_sev.astype(float) / cm_sev.sum(axis=1, keepdims=True)
    labels_sev = [f'Sev{i}' for i in sev_levels]
    annot_sev = np.array([[f'{cm_sev[i,j]}\n({cm_sev_norm[i,j]:.0%})'
                           for j in range(len(sev_levels))]
                          for i in range(len(sev_levels))])
    sns.heatmap(
        cm_sev_norm, annot=annot_sev, fmt='', cmap='Blues',
        xticklabels=labels_sev, yticklabels=labels_sev,
        vmin=0, vmax=1, linewidths=0.5, ax=ax,
    )
    # Highlight Sev3→Sev0 in red
    ax.add_patch(plt.Rectangle((0, 3), 1, 1, fill=False, edgecolor='red', lw=2.5))
    sm = compute_safety_metrics(y_test, y_pred)
    ax.set_title(f'{name}  CMR={sm["cmr"]:.4f}  DR={sm["down_risk"]:.4f}', fontsize=10)
    ax.set_xlabel('Predicted severity'); ax.set_ylabel('True severity')

fig.suptitle('Severity-level confusion — test set', fontsize=11)
plt.tight_layout()
plt.show()

## CV vs Test — Calibration Check

Compare grouped CV estimates (seed=42 and mean across 10 seeds from C4) with test set results.

In [ ]:
# Load seed stability data (C4) for the F1-tuned model
seed_df = pd.read_csv(SEED_CSV_PATH)

print('=== F1-tuned model: CV vs Test calibration ===')
print(f"{'Metric':<20} {'CV seed=42':>12} {'CV mean±std':>18} {'Test':>10}")
print('-' * 65)

cv42 = seed_df[seed_df['Seed'] == 42].iloc[0]

for label, cv_key, test_val in [
    ('Macro F1',    'Macro_F1',   m_f1['macro_f1']),
    ('Balanced Acc','Bal_Acc',    m_f1['balanced_accuracy']),
    ('F06 Recall',  'F06_Recall', m_f1['f06_recall']),
    ('Sev3 Recall', 'Sev3_Recall',m_f1['sev3_recall']),
    ('CMR',         'CMR',        sm_f1['cmr']),
    ('Down Risk',   'Down_Risk',  sm_f1['down_risk']),
]:
    v42  = cv42[cv_key]
    mean = seed_df[cv_key].mean()
    std  = seed_df[cv_key].std()
    print(f"{label:<20} {v42:>12.4f} {mean:>8.4f}±{std:.4f}   {test_val:>10.4f}")

print()
print('Note: CMR and F06 Recall have high CV variability (from C4: CMR CV=18%, F06 CV=41%).')
print('Downward Risk is the most stable CV metric (CV=4.3%).')

## Per-Class Results Table

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

print('=== Per-class results — test set ===')
print()

for name, y_pred in [('F1-tuned', y_pred_f1), ('CMR-tuned', y_pred_cmr)]:
    p, r, f1, sup = precision_recall_fscore_support(
        y_test, y_pred, labels=present_labels, zero_division=0
    )
    print(f'{name}:')
    print(f"  {'Class':<6} {'n':>4} {'Recall':>8} {'Prec':>8} {'F1':>8}")
    print('  ' + '-' * 38)
    for cls, n, rec, pre, f in zip(present_labels, sup, r, p, f1):
        sev = SEVERITY_MAP.get(cls, 0)
        flag = ' ← Sev3' if cls in SEV3 else ''
        print(f"  {cls:<6} {int(n):>4} {rec:>8.3f} {pre:>8.3f} {f:>8.3f}{flag}")
    print()

## Save Results

In [ ]:
results_rows = []
for name, y_pred, m, sm in [
    ('F1-tuned',  y_pred_f1,  m_f1,  sm_f1),
    ('CMR-tuned', y_pred_cmr, m_cmr, sm_cmr),
]:
    row = {'Model': name}
    for k in ['macro_f1','balanced_accuracy','f01_recall','f02_recall','f03_recall',
              'f04_recall','f05_recall','f06_recall','sev3_recall']:
        row[k] = round(m.get(k, float('nan')), 4)
    for k in ['cmr','cmr_count','crit_total','down_risk','sev_err']:
        row[k] = round(sm.get(k, float('nan')), 4) if k not in ('cmr_count','crit_total') else int(sm.get(k, 0))
    results_rows.append(row)

results_df = pd.DataFrame(results_rows)
results_df.to_csv(RESULTS_PATH, index=False)
print(f'Saved: {RESULTS_PATH}')
print()
print(results_df.T.to_string())